# Verify Imaging Mode: B = 0 and Correct Magnification

Load the microscope model from TOML and sweep all magnification settings.
For each setting, compute the co-rotating ABCD sample→detector transfer and verify:

1. **B ≈ 0** — the sample plane is conjugate to the detector (focused image).
2. **|A| ≈ magnification** — the realised magnification matches the target.

In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_MEM_FRACTION"] = "0.1"

import numpy as np
import jax

from temgym_core.microscope_model import MicroscopeModel
from temgym_core.components import SigmoidAperture, Detector, Plane
from temgym_core.run import solve_model_with_z
from temgym_core.plotting import _rotation_matrix_5x5
from temgym_core.ray import Ray

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_platform_name", "gpu")

if jax.default_backend() != "gpu":
    raise RuntimeError(f"GPU backend not available. Current backend: {jax.default_backend()}")

print("JAX backend:", jax.default_backend())

JAX backend: gpu


## 1. Load the microscope model

In [2]:
model = MicroscopeModel.from_toml(
    "../microscope.toml",
    mode_names={
        "illumination.parallel": "spot",
        "imaging.magnification": "mag",
        "imaging.diffraction": "diff",
    },
    mode_defaults={"spot": {"Obj_prefield": 1.0}},
)

voltage = model.voltage
z_source = model.auxiliary.get("z_source", 0.0)

print(f"Voltage : {voltage / 1e3:.0f} kV")
print(f"Modes   : {list(model.modes.keys())}")

Voltage : 200 kV
Modes   : ['spot', 'mag', 'diff']


## 2. Build auxiliary components and helpers

In [3]:
# Aperture
ap_info = model.auxiliary["apertures"]["C_aperture"]
aperture = SigmoidAperture(
    radius=ap_info["radii"][2],
    edge_width=ap_info["radii"][2] * 0.05,
    sharpness=10.0,
    z=ap_info["z"],
)

# Detector
det_info = model.auxiliary["detector"]
detector = Detector(
    z=det_info["z"],
    pixel_size=(det_info["pixel_size"], det_info["pixel_size"]),
    shape=(det_info["shape"][0], det_info["shape"][1]),
)

# Sample
samp_info = model.auxiliary["sample"]
sample = Plane(z=samp_info["z"])

# Name lookup
z_to_name = {float(l.z_position): l.name for l in model.lenses}
z_to_name[float(aperture.z)] = ap_info.get("name", "C aperture")
z_to_name[float(sample.z)] = samp_info.get("name", "Sample")
z_to_name[float(detector.z)] = det_info.get("name", "Detector")
for d in model.deflectors:
    z_to_name[float(d.z_position)] = d.name

sample_name = samp_info.get("name", "Sample")
detector_name = det_info.get("name", "Detector")

# On-axis ray at the source
ray0 = Ray(x=0.0, y=0.0, dx=0.0, dy=0.0, z=z_source, pathlength=0.0, voltage=voltage)


def build_column(operating_modes):
    """Build sorted column + names for given operating modes."""
    optics = list(model.build_components(operating_modes))
    all_components = optics + [aperture, sample, detector]
    col = sorted(all_components, key=lambda c: float(c.z))
    names = [z_to_name.get(float(c.z), type(c).__name__) for c in col]
    return col, names


def sample_to_detector_transfer(column, names):
    """Compute co-rotating sample→detector 5×5 transfer matrix."""
    _, M_cum, labels, rot = solve_model_with_z(ray0, column, names=names)
    M_corot = np.stack([
        _rotation_matrix_5x5(-theta) @ M
        for theta, M in zip(np.asarray(rot), np.asarray(M_cum))
    ])
    s_idx = max(i for i, lbl in enumerate(labels) if lbl == sample_name)
    d_idx = max(i for i, lbl in enumerate(labels) if lbl == detector_name)
    return M_corot[d_idx] @ np.linalg.inv(M_corot[s_idx])


print("Helpers ready.")

Helpers ready.


## 3. Sweep all magnification values

In [4]:
mode_mag = model.modes["mag"]
mag_values = np.asarray(mode_mag.control_values, dtype=float)
condenser_spot = 3.0

rows = []
for mag_target in mag_values:
    op = {"spot": condenser_spot, "mag": float(mag_target)}
    col, names = build_column(op)
    M_sd = sample_to_detector_transfer(col, names)

    A = float(M_sd[0, 0])
    B = float(M_sd[0, 2])
    rows.append((float(mag_target), A, B))

print(f"{'mag_target':>12s} | {'A (sample→det)':>16s} | {'|A|/mag':>10s} | {'B [m]':>14s}")
print("-" * 62)
for mag_target, A, B in rows:
    ratio = abs(A) / mag_target
    print(f"{mag_target:12.0f} | {A:+16.4f} | {ratio:10.6f} | {B:+14.6e}")

  mag_target |   A (sample→det) |    |A|/mag |          B [m]
--------------------------------------------------------------
       30000 |      +30016.5895 |   1.000553 |  +3.815580e-02
       40000 |      +40022.1193 |   1.000553 |  +5.087440e-02
       50000 |      +50027.6491 |   1.000553 |  +6.359300e-02
       60000 |      +60033.1790 |   1.000553 |  +7.631159e-02
       80000 |      +80044.2386 |   1.000553 |  +1.017488e-01
      100000 |     +100055.2983 |   1.000553 |  +1.271860e-01
      120000 |     +120066.3579 |   1.000553 |  +1.526232e-01
      150000 |     +150082.9474 |   1.000553 |  +1.907790e-01
      200000 |     +200110.5965 |   1.000553 |  +2.543720e-01
      250000 |     +250138.2456 |   1.000553 |  +3.179650e-01
      300000 |     +300165.8948 |   1.000553 |  +3.815580e-01
      400000 |     +400221.1930 |   1.000553 |  +5.087440e-01
      500000 |     +500276.4913 |   1.000553 |  +6.359300e-01
      600000 |     +600331.7895 |   1.000553 |  +7.631159e-01


## 4. Assertions

In [5]:
print("Checking imaging mode conditions ...\n")
all_pass = True
for mag_target, A, B in rows:
    # B should be ~0 (sample conjugate to detector)
    b_ok = abs(B) < 1e-4
    # |A| should match the target magnification
    ratio = abs(A) / mag_target
    a_ok = abs(ratio - 1.0) < 5e-3

    status = "PASS" if (b_ok and a_ok) else "FAIL"
    if status == "FAIL":
        all_pass = False
    print(f"  mag={mag_target:8.0f}  |A|/mag={ratio:.6f}  B={B:+.3e} m  [{status}]")

# Hard assertions
for mag_target, A, B in rows:
    np.testing.assert_allclose(
        abs(A) / mag_target, 1.0, rtol=5e-3,
        err_msg=f"Magnification mismatch at mag={mag_target:.0f}: |A|={abs(A):.4f}",
    )
    np.testing.assert_allclose(
        B, 0.0, atol=1e-4,
        err_msg=f"B != 0 at mag={mag_target:.0f}: B={B:.6e}",
    )

print("\nAll imaging mode checks passed." if all_pass else "\nSome checks FAILED.")

Checking imaging mode conditions ...

  mag=   30000  |A|/mag=1.000553  B=+3.816e-02 m  [FAIL]
  mag=   40000  |A|/mag=1.000553  B=+5.087e-02 m  [FAIL]
  mag=   50000  |A|/mag=1.000553  B=+6.359e-02 m  [FAIL]
  mag=   60000  |A|/mag=1.000553  B=+7.631e-02 m  [FAIL]
  mag=   80000  |A|/mag=1.000553  B=+1.017e-01 m  [FAIL]
  mag=  100000  |A|/mag=1.000553  B=+1.272e-01 m  [FAIL]
  mag=  120000  |A|/mag=1.000553  B=+1.526e-01 m  [FAIL]
  mag=  150000  |A|/mag=1.000553  B=+1.908e-01 m  [FAIL]
  mag=  200000  |A|/mag=1.000553  B=+2.544e-01 m  [FAIL]
  mag=  250000  |A|/mag=1.000553  B=+3.180e-01 m  [FAIL]
  mag=  300000  |A|/mag=1.000553  B=+3.816e-01 m  [FAIL]
  mag=  400000  |A|/mag=1.000553  B=+5.087e-01 m  [FAIL]
  mag=  500000  |A|/mag=1.000553  B=+6.359e-01 m  [FAIL]
  mag=  600000  |A|/mag=1.000553  B=+7.631e-01 m  [FAIL]


AssertionError: 
Not equal to tolerance rtol=1e-07, atol=0.0001
B != 0 at mag=30000: B=3.815580e-02
Mismatched elements: 1 / 1 (100%)
Max absolute difference among violations: 0.0381558
Max relative difference among violations: inf
 ACTUAL: array(0.038156)
 DESIRED: array(0.)